#### 1. Loading Data from TensorFlow

`(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()`

- **Meaning:** This line loads the well-known Fashion-MNIST dataset.

- **Structure:** It returns two datasets: one for training (train) and one for testing (test). Each dataset includes an image (`x`) and a numerical label (`y`) representing the clothing types.

#### 2. Creating a "Masking" to Filter Data

`train_mask = np.isin(y_train, [7, 9])

test_mask = np.isin(y_test, [7, 9])`

- **Meaning:** The documentation explains that binary classification only handles 2 classes.

- **Operation:** The `np.isin` function checks if each label in `y_train` and `y_test` belongs to the list `[7, 9]` (Sneaker and Ankle Boot). The result is an array of `True/False` values ​​(masks) so we know which ones to keep.

#### 3. Image Processing (Normalization and Shape Change)

`x_train = x_train[train_mask][..., None]/255.0

x_test = x_test[test_mask][..., None]/255.0`

- **`x_train[train_mask]`**: Only keep images labeled 7 or 9 based on the generated mask.

- **`[..., None]`**: Add a new dimension to the end of the tensor (equivalent to `tf.expand_dims`). This transforms the image from 2D $(28 \times 28)$ to 3D $(28 \times 28 \times 1)$ to meet the requirements of the Convolutional Layer (CNN).

- **`/255.0`**: This is the **Normalization** step. It converts the pixel value from $[0, 255]$ to $[0, 1]$ to stabilize the weight update process and speed up learning.

#### 4. Label Processing (Convert to Binary)

`y_train = (y_train[train_mask] == 9).astype(np.float32)
y_test = (y_test[test_mask] == 9).astype(np.float32)`

- **`(y_train[train_mask] == 9)`**: Converts the original labels to logical values: if it's class 9 (Ankle boot), it's `True`, otherwise it's `False`.

- **`.astype(np.float32)`**: Converts `True/False` to `1.0/0.0`.

- **Reason:** In binary classification, the last class (sigmoid) predicts the probability for class 1 (here defined as class 9). Converting to the `float32` type improves compatibility with matrix operations in TensorFlow.

In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
# Use Fashion-MNIST but map 2 classes to a binary task (e.g., class 9=Ankle boot vs 7=Sneaker)
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
# Keep only classes 7 and 9
train_mask = np.isin(y_train, [7, 9])
test_mask = np.isin(y_test, [7, 9])
x_train = x_train[train_mask][..., None]/255.0
y_train = (y_train[train_mask] == 9).astype(np.float32) # 1 if class 9 else 0
x_test = x_test[test_mask][..., None]/255.0
y_test = (y_test[test_mask] == 9).astype(np.float32)

### 

1. **Conv2D:** Searches for small details (eyes, nose, edges, corners).

2. **MaxPool2D:** Filters to extract the most prominent details and removes unnecessary information.

3. **Dense:** Synthesizes all the prominent information to arrive at a final conclusion.

In [2]:

# 1) CREATE (small CNN)
model = tf.keras.Sequential([
tf.keras.layers.Input(shape=(28,28,1)),
tf.keras.layers.Conv2D(16, 3, activation="relu"),
tf.keras.layers.MaxPool2D(),
tf.keras.layers.Conv2D(32, 3, activation="relu"),
tf.keras.layers.MaxPool2D(),
tf.keras.layers.Flatten(),
tf.keras.layers.Dense(64, activation="relu"),
tf.keras.layers.Dense(1, activation="sigmoid")
])


In [3]:
# 2) COMPILE
model.compile(
loss="binary_crossentropy",
optimizer="adam",
metrics=["accuracy"]
)
# 3) FIT
history = model.fit(x_train, y_train, epochs=5, validation_split=0.1, verbose=0)
# Evaluate
loss, acc = model.evaluate(x_test, y_test, verbose=0)
print("Test accuracy:", acc)

Test accuracy: 0.9649999737739563
